In [1]:
import torch
import dd_solvers

AMGX version 2.5.0
Built on Jul  5 2025, 09:37:04
Compiled with CUDA Runtime 12.6, using CUDA driver 12.9
The AMGX_initialize_plugins API call is deprecated and can be safely removed.


In [2]:
def timeit(fun, warmup_iters=20, iters=100, repetitions=5):
    times = []
    for _ in range(repetitions):
        start = torch.cuda.Event(enable_timing=True)
        end = torch.cuda.Event(enable_timing=True)

        for _ in range(warmup_iters):
            fun()

        start.record()
        for _ in range(iters):
            fun()
        end.record()
        torch.cuda.synchronize()

        times.append(start.elapsed_time(end) / iters)

    return min(times)

In [51]:
n = 1_000_000
k = 4
A = torch.randn((n, k, k), device="cuda", dtype=torch.bfloat16)
b = torch.randn((n, k), device="cuda", dtype=torch.float)
b16 = b.to(torch.bfloat16)

In [ ]:
matmul = torch.ops.dd_solvers_gemv.matmul

In [53]:
exact = (A.to(torch.float64) @ b.to(torch.float64)[:, :, None]).reshape(n, k)

In [54]:
out = matmul(A, b)

In [55]:
torch.linalg.norm(exact - out)

tensor(0.0002, device='cuda:0', dtype=torch.float64)

In [56]:
torch.linalg.norm(exact - (A.to(torch.float) @ b[:, :, None]).reshape(n, k))

tensor(0.0002, device='cuda:0', dtype=torch.float64)

In [57]:
timeit(lambda: matmul(A, b))

0.11259903907775878

In [58]:
timeit(lambda: A @ b16[:, :, None])

1.0854399871826172